# CallWhisper-8k: Hindi-Tuned Hugging Face Models

Purpose: compare the fixed GramVaani slices against Hindi-tuned Whisper-family models. This notebook uses `callwhisper.eval.hf_runner`, which writes the same JSON schema as the OpenAI Whisper runner.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/anshulLuhsna/CallWhisper-8k.git"
REPO_DIR = Path("/content/CallWhisper-8k")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/call-whisper")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"
print("Repo:", Path.cwd())
print("Drive project dir:", DRIVE_PROJECT_DIR)


In [ ]:
!nvidia-smi
!python -m pip install -U pip
!python -m pip install -e ".[dev,colab]"
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

if not DRIVE_PROJECT_DIR.exists():
    raise FileNotFoundError(f"Expected Drive folder not found: {DRIVE_PROJECT_DIR}")

def link_or_replace(source: Path, target: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(f"Missing Drive source: {source}")
    if target.exists() or target.is_symlink():
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)

link_or_replace(DRIVE_PROJECT_DIR / "GV_Dev_5h", REPO_DIR / "datasets/GV_Dev_5h")
link_or_replace(DRIVE_PROJECT_DIR / "Metadata", REPO_DIR / "datasets/Metadata")

drive_manifest_dir = DRIVE_PROJECT_DIR / "manifests"
repo_manifest_dir = REPO_DIR / "datasets/manifests"
if drive_manifest_dir.exists():
    repo_manifest_dir.mkdir(parents=True, exist_ok=True)
    for manifest in drive_manifest_dir.glob("*.csv"):
        shutil.copy2(manifest, repo_manifest_dir / manifest.name)

expected_audio_dir = REPO_DIR / "datasets/GV_Dev_5h/Audio"
print("Drive project dir:", DRIVE_PROJECT_DIR)
print("Audio dir:", expected_audio_dir)
print("Audio exists:", expected_audio_dir.exists())
print("MP3 files:", len(list(expected_audio_dir.glob("*.mp3"))) if expected_audio_dir.exists() else 0)
print("Manifest files:", len(list(repo_manifest_dir.glob("*.csv"))))


## Select Models

Start with one model. If it runs cleanly, expand to the fallback list. Some model cards can change over time; if a model fails to load, record it as blocked rather than spending the whole evening fighting it.

In [ ]:
MODELS = [
    "ARTPARK-IISc/whisper-medium-vaani-hindi",
    # Fallbacks:
    # "vasista22/whisper-hindi-small",
    # "collabora/whisper-base-hindi",
]

MANIFESTS = [
    "datasets/manifests/gramvaani_dev_50.csv",
    "datasets/manifests/gramvaani_dev_50_8khz.csv",
    "datasets/manifests/gramvaani_dev_50_highrate.csv",
]

In [ ]:
def safe_name(model_id: str) -> str:
    return model_id.lower().replace('/', '_').replace('-', '_')

def slice_name(manifest: str) -> str:
    return Path(manifest).stem

for model_id in MODELS:
    for manifest in MANIFESTS:
        output_json = f"results/colab_hf_{safe_name(model_id)}_{slice_name(manifest)}_seed0.json"
        cmd = [
            sys.executable, "-m", "callwhisper.eval.hf_runner",
            "--manifest", manifest,
            "--model-id", model_id,
            "--language-mode", "manifest",
            "--seed", "0",
            "--device", "auto",
            "--output-json", output_json,
        ]
        print("RUN", " ".join(cmd))
        subprocess.run(cmd, check=True, env={**os.environ, "PYTHONPATH": "src"})

In [ ]:
import json, pandas as pd

rows = []
for path in sorted(Path("results").glob("colab_hf_*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    sample0 = payload["samples"][0]
    rows.append({
        "file": str(path),
        "model": sample0["model"],
        "slice": sample0["slice"],
        "files": payload["summary"]["num_files"],
        "wer": round(payload["summary"]["wer"], 4),
        "cer": round(payload["summary"]["cer"], 4),
    })

df = pd.DataFrame(rows).sort_values(["model", "slice"])
df

In [ ]:
Path("results/colab_hf_model_summary.md").write_text(df.to_markdown(index=False) + "\n", encoding="utf-8")
print(Path("results/colab_hf_model_summary.md").read_text(encoding="utf-8"))